# Session 3: XGBoost & SHAP for Wheat Yield Prediction

**Duration:** 2 hours

**Learning Objectives:**
- Understand the difference between bagging (RF) and boosting (XGBoost)
- Train XGBoost with early stopping
- Use SHAP to explain individual predictions
- Compare RF vs XGBoost with same features

**Session plan:**
1. Setup (same data as Sessions 1–2)
2. **Model 1:** XGBoost with same 6 features → compare with RF (Session 2)
3. **Model 2:** XGBoost with monthly features → compare with RF monthly
4. SHAP explanations
5. Final model comparison
6. Summary

---

## Part 1: Setup (10 min)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
import time
import warnings
warnings.filterwarnings('ignore')

print(f"XGBoost version: {xgb.__version__}")

# ── Load data (same as Sessions 1-2) ──
df = pd.read_parquet("wa_features_1989-2020.parquet")
df = df[df['wheat_yield'] > 0].copy()

print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Years: {df['year'].min()}-{df['year'].max()}")

In [ ]:
# ── Same setup as Session 2 ──

# Metrics (same as Sessions 1-2)
def concordance_cc(actual, predicted):
    mean_act  = np.mean(actual)
    mean_pred = np.mean(predicted)
    var_act   = np.var(actual)
    var_pred  = np.var(predicted)
    covariance = np.mean((actual - mean_act) * (predicted - mean_pred))
    return (2 * covariance) / (var_act + var_pred + (mean_act - mean_pred)**2)

def print_metrics(name, actual, predicted):
    r2   = r2_score(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    bias = np.mean(predicted - actual)
    lccc = concordance_cc(actual, predicted)
    print(f"  {name:<30s} R2={r2:.3f}  RMSE={rmse:.3f}  MAE={mae:.3f}  Bias={bias:+.3f}  LCCC={lccc:.3f}")
    return {"name": name, "R2": r2, "RMSE": rmse, "MAE": mae, "Bias": bias, "LCCC": lccc}

# ── 6 simple features (same as Sessions 1-2) ──
FEATURE_COLS = ['rain_preseason', 'rain_early', 'rain_late',
                'gdd_may_nov', 'frost_days_aug_oct', 'heat_days_aug_oct']

# ── Monthly features (same as Session 2) ──
monthly_rain = [c for c in df.columns if c.startswith("rain_m") or c.startswith("rain_mpre")]
monthly_rad  = [c for c in df.columns if c.startswith("rad_sum_m")]
monthly_fasw = [c for c in df.columns if c.startswith("fasw_mean_m") and "aug" not in c]
monthly_gdd  = [c for c in df.columns if c.startswith("gdd_m")]
soil_feats   = ["pawc_0_30_mm", "ph_0_30"]
augoct_feats = [c for c in ["frost_days_aug_oct","heat_days_aug_oct",
                             "rad_aug_oct","fasw_mean_aug_oct"] if c in df.columns]

MONTHLY_FEATURES = soil_feats + monthly_rain + monthly_rad + monthly_fasw + monthly_gdd + augoct_feats
MONTHLY_FEATURES = [f for f in MONTHLY_FEATURES if f in df.columns]

# ── Temporal split (same as Sessions 1-2) ──
train = df[df['year'] <= 2018]
test  = df[df['year'] >= 2019]

actual_test = test['wheat_yield'].values

print(f"Simple features:  {len(FEATURE_COLS)}")
print(f"Monthly features: {len(MONTHLY_FEATURES)}")
print(f"Train: {train.year.min()}-{train.year.max()} ({len(train):,} rows)")
print(f"Test:  {test.year.min()}-{test.year.max()}  ({len(test):,} rows)")

## Part 2: Model 1 — XGBoost with 6 Features (25 min)

### Bagging vs Boosting — Quick Recap

| | Random Forest (Bagging) | XGBoost (Boosting) |
|---|---|---|
| **Strategy** | Train trees independently, average them | Train trees sequentially, each fixes previous errors |
| **Each tree fits** | Raw yield | Residuals from all previous trees |
| **Combining** | Average | Sum of corrections |
| **Main goal** | Reduce variance | Reduce bias |

Let's see if XGBoost can beat RF using the same 6 features:


In [ ]:
# ── Prepare 6-feature data ──
X_train_6 = train[FEATURE_COLS].values.astype(np.float32)
X_test_6  = test[FEATURE_COLS].values.astype(np.float32)
y_train   = train['wheat_yield'].values.astype(np.float32)

# ── RF baseline from Session 2 (reproduce) ──
rf_6 = RandomForestRegressor(n_estimators=500, max_depth=15, min_samples_leaf=10,
                              max_features=0.5, random_state=42, n_jobs=-1)
rf_6.fit(X_train_6, y_train)
pred_rf_6 = rf_6.predict(X_test_6)

print("=== Session 2 baseline ===")
m_rf6 = print_metrics("RF (6 features)", actual_test, pred_rf_6)

In [ ]:
# ── Basic XGBoost (no tuning) ──
t0 = time.time()
xgb_basic = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)
xgb_basic.fit(X_train_6, y_train)
dt = time.time() - t0

pred_xgb_basic = xgb_basic.predict(X_test_6)

print(f"=== XGBoost basic (6 features, {dt:.1f}s) ===")
m_xgb_basic = print_metrics("XGBoost basic (6 feat)", actual_test, pred_xgb_basic)

### Early Stopping — Knowing When to Stop

XGBoost adds trees sequentially. At some point, more trees = overfitting.
**Early stopping** watches a validation set and stops when it stops improving.


In [ ]:
# ── XGBoost with early stopping ──
# Split training into train + validation for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_6, y_train, test_size=0.15, random_state=42
)

xgb_es_6 = xgb.XGBRegressor(
    n_estimators=2000,          # set high — early stopping will find the right number
    max_depth=6,
    learning_rate=0.05,         # smaller = more trees but better generalisation
    subsample=0.8,              # use 80% of rows per tree
    colsample_bytree=0.8,       # use 80% of features per tree
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50,
)
xgb_es_6.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

pred_xgb_es_6 = xgb_es_6.predict(X_test_6)

print(f"Best iteration: {xgb_es_6.best_iteration} out of 2000")
print(f"(Early stopping saved {2000 - xgb_es_6.best_iteration} unnecessary trees)")
print()
m_xgb_es_6 = print_metrics("XGBoost+ES (6 feat)", actual_test, pred_xgb_es_6)

In [ ]:
# ── Learning curve ──
results = xgb_es_6.evals_result()
val_rmse = results["validation_0"]["rmse"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(val_rmse, color="tab:red", linewidth=1)
ax.axvline(xgb_es_6.best_iteration, color="black", linestyle="--",
           label=f"Best: {xgb_es_6.best_iteration} trees")
ax.set_xlabel("Boosting Rounds")
ax.set_ylabel("Validation RMSE (t/ha)")
ax.set_title("XGBoost Learning Curve — Early Stopping")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Compare RF vs XGBoost (6 features) ──
print("=" * 75)
print("MODEL 1: Same 6 features — RF vs XGBoost")
print("=" * 75)
print(f"{'Model':<30s} {'R2':>6s} {'RMSE':>7s} {'MAE':>7s} {'Bias':>7s} {'LCCC':>6s}")
print("-" * 75)
for m in [m_rf6, m_xgb_basic, m_xgb_es_6]:
    print(f"{m['name']:<30s} {m['R2']:6.3f} {m['RMSE']:7.3f} {m['MAE']:7.3f} {m['Bias']:+7.3f} {m['LCCC']:6.3f}")

**Discussion:**
1. Does XGBoost beat RF with the same 6 features?
2. Did early stopping help? Compare basic vs early-stopping XGBoost.
3. Look at the learning curve — where would overfitting start without early stopping?


## Part 3: Model 2 — Monthly Features (25 min)

Now let's give both RF and XGBoost the full monthly feature set and compare.


In [ ]:
# ── Prepare monthly data ──
X_train_m = train[MONTHLY_FEATURES].values.astype(np.float32)
X_test_m  = test[MONTHLY_FEATURES].values.astype(np.float32)

# Fill NaN
for i in range(X_train_m.shape[1]):
    med = np.nanmedian(X_train_m[:, i])
    X_train_m[np.isnan(X_train_m[:, i]), i] = med
    X_test_m[np.isnan(X_test_m[:, i]), i] = med

# ── RF with monthly features (reproduce Session 2) ──
rf_m = RandomForestRegressor(n_estimators=500, max_depth=15, min_samples_leaf=10,
                              max_features=0.5, random_state=42, n_jobs=-1)
rf_m.fit(X_train_m, y_train)
pred_rf_m = rf_m.predict(X_test_m)

print("=== RF with monthly features (Session 2 result) ===")
m_rf_m = print_metrics(f"RF ({len(MONTHLY_FEATURES)} feat)", actual_test, pred_rf_m)

In [ ]:
# ── XGBoost with monthly features + early stopping ──
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_train_m, y_train, test_size=0.15, random_state=42
)

xgb_es_m = xgb.XGBRegressor(
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50,
)
xgb_es_m.fit(X_tr_m, y_tr_m, eval_set=[(X_val_m, y_val_m)], verbose=False)

pred_xgb_m = xgb_es_m.predict(X_test_m)

print(f"Best iteration: {xgb_es_m.best_iteration}")
m_xgb_m = print_metrics(f"XGBoost+ES ({len(MONTHLY_FEATURES)} feat)", actual_test, pred_xgb_m)

In [ ]:
# ── Full comparison: 6 feat vs monthly, RF vs XGBoost ──
print("=" * 80)
print("FULL COMPARISON: RF vs XGBoost x Simple vs Monthly features")
print("=" * 80)
print(f"{'Model':<35s} {'#Feat':>5s} {'R2':>6s} {'RMSE':>7s} {'MAE':>7s} {'Bias':>7s} {'LCCC':>6s}")
print("-" * 80)
for m, nf in [(m_rf6, 6), (m_xgb_es_6, 6), (m_rf_m, len(MONTHLY_FEATURES)), (m_xgb_m, len(MONTHLY_FEATURES))]:
    print(f"{m['name']:<35s} {nf:>5d} {m['R2']:6.3f} {m['RMSE']:7.3f} {m['MAE']:7.3f} {m['Bias']:+7.3f} {m['LCCC']:6.3f}")
print()
print(f"RF:  6 feat -> monthly:  R2 {m_rf_m['R2']-m_rf6['R2']:+.3f}  RMSE {m_rf_m['RMSE']-m_rf6['RMSE']:+.3f}")
print(f"XGB: 6 feat -> monthly:  R2 {m_xgb_m['R2']-m_xgb_es_6['R2']:+.3f}  RMSE {m_xgb_m['RMSE']-m_xgb_es_6['RMSE']:+.3f}")
print(f"Monthly: RF -> XGB:      R2 {m_xgb_m['R2']-m_rf_m['R2']:+.3f}  RMSE {m_xgb_m['RMSE']-m_rf_m['RMSE']:+.3f}")

In [ ]:
# ── Scatter plots: RF vs XGBoost (monthly features) ──
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (name, pred, r2) in zip(axes, [
    (f"RF ({len(MONTHLY_FEATURES)} feat)", pred_rf_m, m_rf_m['R2']),
    (f"XGBoost ({len(MONTHLY_FEATURES)} feat)", pred_xgb_m, m_xgb_m['R2']),
]):
    ax.scatter(actual_test, pred, alpha=0.05, s=3, color='steelblue')
    lim = max(actual_test.max(), pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='1:1 line')
    ax.set_xlabel('Actual yield (t/ha)')
    ax.set_ylabel('Predicted yield (t/ha)')
    ax.set_title(f'{name}\nR2={r2:.3f}')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-year comparison ──
test_years = test['year'].values
print("=== Per-year: RF vs XGBoost (monthly features) ===")
for yr in [2019, 2020]:
    m = test_years == yr
    print(f"\n  Year {yr} (n={m.sum():,}):")
    for name, pred in [(f"RF ({len(MONTHLY_FEATURES)})", pred_rf_m),
                        (f"XGB ({len(MONTHLY_FEATURES)})", pred_xgb_m)]:
        r2 = r2_score(actual_test[m], pred[m])
        rmse = np.sqrt(mean_squared_error(actual_test[m], pred[m]))
        bias = np.mean(pred[m] - actual_test[m])
        print(f"    {name:<25s} R2={r2:.3f}  RMSE={rmse:.3f}  Bias={bias:+.3f}")

## Part 4: SHAP Explanations (30 min)

SHAP values explain **each individual prediction**: for every grid cell, how much
did each feature push the yield prediction up or down from the average?

This is essential for scientific applications — we need to know *why* the model
predicts what it predicts, not just that it's accurate.


In [ ]:
import shap

# ── SHAP for XGBoost (monthly features) ──
explainer = shap.TreeExplainer(xgb_es_m)

# Subsample test set for speed
n_shap = min(5000, len(X_test_m))
rng = np.random.RandomState(42)
shap_idx = rng.choice(len(X_test_m), n_shap, replace=False)
X_shap = X_test_m[shap_idx]

print(f"Computing SHAP values for {n_shap} samples...")
shap_values = explainer.shap_values(X_shap)
print(f"Done! Shape: {shap_values.shape}")

In [ ]:
# ── Global importance: bar plot ──
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=MONTHLY_FEATURES,
                  plot_type="bar", max_display=20, show=False)
plt.title("XGBoost SHAP: Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ── Beeswarm: direction of effect ──
# Red = high feature value, Blue = low
# Right = pushes yield UP, Left = pushes yield DOWN

fig = plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, feature_names=MONTHLY_FEATURES,
                  max_display=20, show=False)
plt.title("How Each Feature Affects Yield")
plt.tight_layout()
plt.show()

print("Reading the beeswarm:")
print("  Red dots  = high values of that feature")
print("  Blue dots = low values of that feature")
print("  Right     = pushes yield UP")
print("  Left      = pushes yield DOWN")

In [ ]:
# ── Waterfall: explain individual predictions ──
# Pick one high-yield and one low-yield cell

high_idx = np.argmax(actual_test[shap_idx])
low_idx  = np.argmin(actual_test[shap_idx])

print(f"=== High yield cell ===")
print(f"Actual: {actual_test[shap_idx[high_idx]]:.2f} t/ha   "
      f"Predicted: {pred_xgb_m[shap_idx[high_idx]]:.2f} t/ha")

fig = plt.figure(figsize=(14, 4))
shap.waterfall_plot(
    shap.Explanation(values=shap_values[high_idx],
                     base_values=explainer.expected_value,
                     data=X_shap[high_idx],
                     feature_names=MONTHLY_FEATURES),
    max_display=12, show=False)
plt.title("High Yield Cell: What Drove This Prediction?")
plt.tight_layout()
plt.show()

In [ ]:
print(f"=== Low yield cell ===")
print(f"Actual: {actual_test[shap_idx[low_idx]]:.2f} t/ha   "
      f"Predicted: {pred_xgb_m[shap_idx[low_idx]]:.2f} t/ha")

fig = plt.figure(figsize=(14, 4))
shap.waterfall_plot(
    shap.Explanation(values=shap_values[low_idx],
                     base_values=explainer.expected_value,
                     data=X_shap[low_idx],
                     feature_names=MONTHLY_FEATURES),
    max_display=12, show=False)
plt.title("Low Yield Cell: What Drove This Prediction?")
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP dependence plot: explore one feature ──
# Shows how a single feature's value relates to its SHAP value

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, ["rain_m2", "fasw_mean_m4", "gdd_m3"]):
    if feat in MONTHLY_FEATURES:
        fi = MONTHLY_FEATURES.index(feat)
        ax.scatter(X_shap[:, fi], shap_values[:, fi], alpha=0.1, s=3, color='steelblue')
        ax.axhline(0, color='red', linestyle='--', alpha=0.5)
        ax.set_xlabel(feat)
        ax.set_ylabel(f"SHAP value (t/ha)")
        ax.set_title(f"Effect of {feat} on yield")
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("These show the non-linear relationships that XGBoost learned:")
print("  - The curve shape shows how the feature affects yield")
print("  - Flat regions = feature doesn't matter at those values")
print("  - Steep regions = feature is critical")

**Discussion:**
1. Look at the beeswarm — which features have the clearest direction of effect?
2. In the waterfall plots, what drove the high-yield vs low-yield predictions?
3. Do the SHAP results match your agronomic understanding of WA wheat?
4. Which months matter most for rainfall? For soil moisture?


## Part 5: Learning Rate and Depth Effects (15 min)

The two most important XGBoost hyperparameters:
- **learning_rate** ($\eta$): how much each tree contributes (smaller = more robust)
- **max_depth**: tree complexity (shallower = less overfitting)


In [ ]:
# ── Effect of learning rate ──
lr_results = []
for lr in [0.01, 0.05, 0.1, 0.2, 0.3]:
    model = xgb.XGBRegressor(
        n_estimators=2000, max_depth=6, learning_rate=lr,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        random_state=42, n_jobs=-1, tree_method="hist",
        early_stopping_rounds=50)
    model.fit(X_tr_m, y_tr_m, eval_set=[(X_val_m, y_val_m)], verbose=False)
    pred = model.predict(X_test_m)
    lr_results.append({
        "learning_rate": lr,
        "best_trees": model.best_iteration,
        "R2": r2_score(actual_test, pred),
        "RMSE": np.sqrt(mean_squared_error(actual_test, pred))
    })

lr_df = pd.DataFrame(lr_results)
print(lr_df.to_string(index=False))
print("\nSmaller learning rate → more trees needed, but often better results.")

In [ ]:
# ── Effect of max_depth ──
depth_results = []
for d in [3, 4, 5, 6, 8, 10]:
    model = xgb.XGBRegressor(
        n_estimators=2000, max_depth=d, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        random_state=42, n_jobs=-1, tree_method="hist",
        early_stopping_rounds=50)
    model.fit(X_tr_m, y_tr_m, eval_set=[(X_val_m, y_val_m)], verbose=False)
    pred = model.predict(X_test_m)
    depth_results.append({
        "max_depth": d,
        "best_trees": model.best_iteration,
        "R2": r2_score(actual_test, pred),
        "RMSE": np.sqrt(mean_squared_error(actual_test, pred))
    })

depth_df = pd.DataFrame(depth_results)
print(depth_df.to_string(index=False))
print("\nDepth 4-6 is usually the sweet spot for tabular data.")

## Part 6: Final Model Comparison Across All Sessions (10 min)

In [ ]:
# ── Summary table across all sessions ──
print("=" * 85)
print("FINAL COMPARISON: All models across Sessions 1-3")
print("=" * 85)
print(f"{'Session':<10s} {'Model':<35s} {'#Feat':>5s} {'R2':>6s} {'RMSE':>7s} {'Bias':>7s} {'LCCC':>6s}")
print("-" * 85)
for sess, m, nf in [
    ("S2", m_rf6, 6),
    ("S2", m_rf_m, len(MONTHLY_FEATURES)),
    ("S3", m_xgb_es_6, 6),
    ("S3", m_xgb_m, len(MONTHLY_FEATURES)),
]:
    print(f"{sess:<10s} {m['name']:<35s} {nf:>5d} {m['R2']:6.3f} {m['RMSE']:7.3f} {m['Bias']:+7.3f} {m['LCCC']:6.3f}")

print()
print("Key insights:")
print(f"  Better features (6 -> monthly):  RF +{m_rf_m['R2']-m_rf6['R2']:.3f}   XGB +{m_xgb_m['R2']-m_xgb_es_6['R2']:.3f}")
print(f"  Better algorithm (RF -> XGB):    6feat +{m_xgb_es_6['R2']-m_rf6['R2']:.3f}  monthly +{m_xgb_m['R2']-m_rf_m['R2']:.3f}")

In [ ]:
# ── Four-panel scatter ──
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for ax, (name, pred, r2) in zip(axes.flat, [
    ("RF (6 feat)", pred_rf_6, m_rf6['R2']),
    ("XGBoost (6 feat)", pred_xgb_es_6, m_xgb_es_6['R2']),
    (f"RF ({len(MONTHLY_FEATURES)} feat)", pred_rf_m, m_rf_m['R2']),
    (f"XGBoost ({len(MONTHLY_FEATURES)} feat)", pred_xgb_m, m_xgb_m['R2']),
]):
    ax.scatter(actual_test, pred, alpha=0.05, s=3, color='steelblue')
    lim = max(actual_test.max(), pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5)
    ax.set_xlabel('Actual yield (t/ha)')
    ax.set_ylabel('Predicted yield (t/ha)')
    ax.set_title(f'{name}\nR2={r2:.3f}')
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)

plt.suptitle("RF vs XGBoost x Simple vs Monthly", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Summary

### What we learned

| | 6 Seasonal Features | Monthly Features |
|---|---|---|
| **Random Forest** | Session 2 baseline | Session 2 best |
| **XGBoost** | Comparable to RF | Best overall (with SHAP!) |

### Key Takeaways
1. **XGBoost ≈ RF** with the same features — the real gain comes from better features
2. **Early stopping is essential** for XGBoost — prevents overfitting
3. **SHAP values** explain individual predictions — critical for scientific credibility
4. **Monthly features >> seasonal aggregates** — timing of rain/stress matters most
5. **Neither model can extrapolate** beyond the training data range

### Exercises
1. Use `shap.dependence_plot` to explore how `rain_m0` (sowing month rain) affects yield. Is there a saturation point?
2. Compare SHAP values for 2019 vs 2020 separately. Do the important features change between years?
3. Try `max_depth=4` with `learning_rate=0.01` and 5000 estimators. Does a shallower, slower model do better?
4. What are the limitations of these models for predicting yield under future climate conditions?
